# Reviews

Address comments.

In [2]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder

In [4]:
import nbimporter # lets you import notebooks like regular modules

grids_module = __import__('3_grids')
mature_module = __import__('6_mature')
%run 7_write_csv.ipynb # to make unified_data available here

In [6]:
# Load the categorical image and select the 'biome' band
biomes = ee.Image(f"{data_folder}/categorical").select("biome")
biomes_mask = biomes.eq(1).Or(biomes.eq(4)).rename("biome_mask")

amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))

age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").rename("age")

biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').mean().select("AGB").rename("biomass")
sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

lulc = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))
mature_mask = lulc.eq(3).reduce(ee.Reducer.allNonZero()).selfMask().updateMask(biomes_mask)

## GEDI - mean biomass per 10km grid cell

Aggregating GEDI L4A into 10km pixels doesn't work as fast and as well with reduceResolution + reproject. That process ends up running out of computational power.

In order to avoid this, it is necessary to get the mean per 10km grid cell over many tiles.


In [ ]:
def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2015-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI = GEDI.mean().toInt16().rename('GEDI_biomass')

GEDI_mature = GEDI.updateMask(mature_mask).rename("GEDI_mature_biomass")

GEDI_mature = GEDI_mature.updateMask(distance_gt_1000).clip(amazon)

grid = amazon.geometry().coveringGrid('EPSG:4326', 200000)  # 200km tiles

# Convert to a list so we can iterate
grid_list = grid.toList(grid.size())
n_tiles = grid.size().getInfo()
print(f"Number of tiles: {n_tiles}")

for i in range(n_tiles):
    tile = ee.Feature(grid_list.get(i)).geometry()
    
    # Aggregate within this tile only
    tile_image = GEDI_mature \
        .setDefaultProjection(crs='EPSG:4326', scale = 25) \
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=1024,
            bestEffort=True
        ) \
        .reproject(crs='EPSG:4326', scale = 10000) \
        .clip(tile)
    
    task = ee.batch.Export.image.toAsset(
        image=tile_image,
        description=f"GEDI_mature_biomass_10k_tile_{i}",
        assetId=f"{data_folder}/GEDI_mature/GEDI_mature_biomass_10k_tile_{i}",
        region=tile,
        crs='EPSG:4326',
        scale=10000,
        maxPixels=1e13
    )
    task.start()
    print(f"Started tile {i}/{n_tiles}")




### GEDI nearest neighbor
After the GEDI data is aggregated to 10km resolution, we use the same method from 6_mature to get the nearest mature biomass for the gaps in the image (areas where there was no GEDI read on mature forests in the 10km grid cell pixel)

In [ ]:
obtain_nearest_mature_neighbor = mature_module.obtain_nearest_mature_neighbor

all_images = utils.import_folder_features(f"{data_folder}/GEDI_mature", asset_type='image')
mature_biomass_10k = ee.ImageCollection(all_images).mosaic()

features_secondary = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed")

# obtain_nearest_mature_neighbor(mature_biomass_10k, features_secondary, "_GEDI")



### Grid sampling - GEDI

Use the same sampling method to obtain one pixel of secondary forest that coincides with a GEDI footprint per 10km2



In [ ]:

age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").rename("age")

export_image = age.updateMask(edge)

def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2020-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI = GEDI.mean().toInt16().rename('GEDI_biomass').reproject(age.projection())

export_image = export_image.addBands(GEDI).updateMask(GEDI)

export_image = export_image.select('age').selfMask()

# create_grid(export_image, region_name = "amazon", cell_size = 10000, file_name = "secondary_edge_removed_gedi")




In [ ]:

def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2020-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI = GEDI.mosaic().toInt16().rename('biomass')
GEDI_reproj = GEDI.reproject(age.projection())

GEDI_mask = GEDI_reproj.gt(0).selfMask().rename("mask")

unified_data = ee.Image.cat([age, GEDI_reproj, biomass.rename("ESA_biomass"), GEDI_mask])


grid = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed_gedi")


# export_csv("secondary", unified_data_secondary, 10, n_chunks = 30)



## Same-age patches

Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again

### Export age and biomass for ESA CCI for the same-age patches

In [ ]:
# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    task = ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    )
    # task.start()
